# Exercise: extending a power network model

No advanced knowledge of power systems is required. All you need to know:

- The network is a **tree** starting from a source substation. Each **node** consumes and/or generates power, in MW.
- The power flowing through a **line** equals the net demand (load − generation) of everything downstream of that line.
  It can be **negative** when downstream generation exceeds the load: power then flows back towards the source.
- Each line has a maximum **capacity** (in MW), whatever the direction of the flow.

Provided files, as cells below: `network.py` (the model), `example.py` (a small runnable network), `test_network.py` (tests, run with `pytest`).

## Using this notebook

1. First run all cells once (*Runtime > Run all* in Colab, *Run > Run All Cells* in Jupyter).
2. Each cell starting with `%%writefile` writes a Python file to disk. **To change the code, edit the cell directly, then run it again** (Shift + Enter): otherwise the file is not updated.
3. Then run the test cell again (`!python -m pytest`).

Feel free to add cells for your own experiments.

## Step 1 — Getting started and fixing a bug

Run the tests. One of them fails. Fix it.

## Step 2 — Taking a line out of service

Operators need to simulate the loss of a line (fault or maintenance). Add:

- `take_out_of_service(line_name)` and `put_back_in_service(line_name)`;

Business rule: a node that is no longer connected to the source neither consumes nor generates anything (generators disconnect automatically for
safety reasons).
The existing methods must remain consistent with the state of the network.

Moreover, we want to be able to compute the set of nodes no longer connected to the source and the total load of the de-energized nodes. 

## Step 3 — N-1 analysis

Add `n_minus_1_analysis()` which, for each line in service, simulates its loss and returns the corresponding unserved
load. The network must be in the same state after the call as before.


## Setup (harmless if pytest is already installed)

In [ ]:
%pip install -q pytest

## The model: `network.py`

In [ ]:
%%writefile network.py
"""Simplified model of a radial electrical distribution network.

A radial network is a tree: it starts at a source substation and branches
out towards consumers, without any loop. Each line therefore has an
"upstream" side (towards the source) and a "downstream" side (towards the ends).
"""
from dataclasses import dataclass


@dataclass
class Node:
    name: str
    load_mw: float = 0.0
    generation_mw: float = 0.0  # e.g. solar panels, wind turbine

    @property
    def net_demand_mw(self) -> float:
        """Positive when the node consumes more than it generates."""
        return self.load_mw - self.generation_mw


@dataclass
class Line:
    name: str
    upstream: str    # node on the source side
    downstream: str  # node on the far end side
    capacity_mw: float


class Network:
    def __init__(self, source: str):
        self.source = source
        self.nodes: dict[str, Node] = {source: Node(source)}
        self.lines: dict[str, Line] = {}

    def add_node(self, name, load_mw=0.0, generation_mw=0.0):
        if name in self.nodes:
            raise ValueError(f"Node {name} already exists")
        self.nodes[name] = Node(name, load_mw, generation_mw)

    def add_line(self, name, upstream, downstream, capacity_mw):
        for n in (upstream, downstream):
            if n not in self.nodes:
                raise ValueError(f"Unknown node: {n}")
        self.lines[name] = Line(name, upstream, downstream, capacity_mw)

    def _downstream_lines(self, node):
        return [l for l in self.lines.values() if l.upstream == node]

    def _downstream_demand(self, node):
        """Net demand of the node and of everything downstream of it."""
        total = self.nodes[node].net_demand_mw
        for line in self._downstream_lines(node):
            total += self._downstream_demand(line.downstream)
        return total

    def compute_flows(self):
        """Power flow (MW) in each line. Positive = from upstream to downstream."""
        return {name: self._downstream_demand(l.downstream) for name, l in self.lines.items()}

    def overloaded_lines(self):
        flows = self.compute_flows()
        return [name for name, f in flows.items() if f > self.lines[name].capacity_mw]

    def source_balance_mw(self):
        """Power supplied by the source substation (negative = export upstream)."""
        return self._downstream_demand(self.source)

## Example network: `example.py`

In [ ]:
%%writefile example.py
"""Example network used by the tests.

    SOURCE --L1--> A --L2--> B
                   |
                   +--L3--> C --L4--> D
"""
from network import Network


def build_example_network():
    net = Network("SOURCE")
    net.add_node("A", load_mw=3.0)
    net.add_node("B", load_mw=2.0)
    net.add_node("C", load_mw=1.0)
    net.add_node("D", load_mw=1.5)
    net.add_line("L1", "SOURCE", "A", capacity_mw=10.0)
    net.add_line("L2", "A", "B", capacity_mw=6.0)
    net.add_line("L3", "A", "C", capacity_mw=4.0)
    net.add_line("L4", "C", "D", capacity_mw=3.0)
    return net


if __name__ == "__main__":
    network = build_example_network()
    for name, f in network.compute_flows().items():
        print(f"{name}: {f:.2f} MW / {network.lines[name].capacity_mw} MW")
    print("Source balance:", network.source_balance_mw(), "MW")
    print("Overloaded lines:", network.overloaded_lines())

In [ ]:
!python example.py

## Tests: `test_network.py`

In [ ]:
%%writefile test_network.py
import pytest

from example import build_example_network
from network import Network


def test_flows_example_network():
    flows = build_example_network().compute_flows()
    assert flows == pytest.approx({"L1": 7.5, "L2": 2.0, "L3": 2.5, "L4": 1.5})


def test_source_balance():
    assert build_example_network().source_balance_mw() == pytest.approx(7.5)


def test_overload_when_consuming():
    net = build_example_network()
    net.lines["L4"].capacity_mw = 1.0
    assert net.overloaded_lines() == ["L4"]


def test_overload_when_injecting():
    net = Network("SOURCE")
    net.add_node("PV_FARM", load_mw=0.5, generation_mw=8.0)
    net.add_line("L1", "SOURCE", "PV_FARM", capacity_mw=5.0)
    assert net.compute_flows()["L1"] == pytest.approx(-7.5)
    assert net.overloaded_lines() == ["L1"]


def test_unknown_node():
    net = Network("SOURCE")
    with pytest.raises(ValueError):
        net.add_line("L1", "SOURCE", "X", capacity_mw=1.0)

## Run the tests

In [ ]:
!python -m pytest -q